## lab21-day07 — chạy tổng trên Kaggle
Settings: **Accelerator = GPU**, **Internet = On**. Input: dataset `lab21-data` và `lab21-scripts`. Rồi **Run All**.

Luồng chạy: cài đặt -> kiểm tra dữ liệu -> **quét tracker/conf/iou** (video_1 chấm bằng HOTA, video_2..5 thống kê) -> chạy bản nộp đủ frame -> chấm video_1 -> nén và tải về.
Thời gian ước tính trên T4: cài đặt ~7 phút, quét video_1 ~30 phút, quét video_2..5 ~35 phút, bản nộp ~10 phút (tổng ~1,5 giờ). Đặt `RUN_SWEEP_* = False` để bỏ qua quét. Cấu hình nộp lấy từ kết quả quét (`best_config.json`); `SUBMISSION_CONFIG` trong `run_all.py` chỉ là dự phòng.

Venv Python 3.11 vì Kaggle chạy Python 3.13, `lap`/`filterpy` không build được trên đó.

In [ ]:
# 1. Tạo venv Python 3.11 bằng uv, cài đúng gói lab + TrackEval
import os, subprocess, sys
!pip install -q uv
!rm -rf /kaggle/working/venv /kaggle/working/TrackEval
!uv venv --python 3.11 /kaggle/working/venv
PY = "/kaggle/working/venv/bin/python"
os.environ["PY"] = PY

# torch bản CUDA 12.1 (chạy được với driver của Kaggle), rồi mới tới các gói còn lại
!uv pip install -q --python $PY torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python $PY "ultralytics>=8.4" boxmot==10.0.42 lap filterpy opencv-python-headless scipy matplotlib pyyaml 'setuptools<81' wheel
!git clone -q https://github.com/JonathonLuiten/TrackEval.git /kaggle/working/TrackEval
!uv pip install -q --python $PY -e /kaggle/working/TrackEval
!$PY -c "import ultralytics, boxmot, trackeval, torch; print('OK ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# 2. Chép script, tìm dữ liệu, tự dò LAB_DATA
import glob, shutil
from pathlib import Path

DATA_DS = "/kaggle/input/datasets/nguyenthanhhoak18hcm/lab21-data"
SCRIPT_DS = "/kaggle/input/datasets/nguyenthanhhoak18hcm/lab21-scripts"
WORK = Path("/kaggle/working")
os.chdir(WORK)

(WORK / "scripts").mkdir(exist_ok=True)
found = glob.glob(f"{SCRIPT_DS}/**/run_tracking.py", recursive=True)
assert found, f"Không thấy run_tracking.py trong {SCRIPT_DS}"
for f in Path(found[0]).parent.glob("*.py"):
    shutil.copy(f, WORK / "scripts" / f.name)
print("scripts:", sorted(p.name for p in (WORK / "scripts").iterdir()))

def find_lab_data(root):
    hits = glob.glob(f"{root}/**/video_1/img1", recursive=True)
    return str(Path(hits[0]).parent.parent) if hits else None

lab_data = find_lab_data(DATA_DS)
if lab_data is None:
    zips = glob.glob(f"{DATA_DS}/**/*.zip", recursive=True)
    assert zips, f"Không thấy .zip hay video_1/img1 trong {DATA_DS}"
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", str(WORK / "lab_raw")], check=True)
    lab_data = find_lab_data(str(WORK / "lab_raw"))
assert lab_data, "Không dò được thư mục chứa video_1 … video_5"
os.environ["LAB_DATA"] = lab_data
print("LAB_DATA =", lab_data)
print("Nội dung video_1:", sorted(p.name for p in Path(lab_data, "video_1").iterdir()))

# GPU nếu có, không thì CPU (rất chậm)
has_cuda = subprocess.run([PY, "-c", "import torch,sys; sys.exit(0 if torch.cuda.is_available() else 1)"]).returncode == 0
os.environ["DEVICE"] = "cuda:0" if has_cuda else "cpu"
print("DEVICE =", os.environ["DEVICE"])

In [ ]:
# 3. Kiểm tra dữ liệu: cả 5 video phải OK, chỉ video_1 có nhãn
!$PY scripts/check_data.py --lab-data-root "$LAB_DATA"

Hai cell dưới ghi đè `run_all.py` (thêm `--config-json`) và thêm `sweep.py`, nên không cần upload lại dataset `lab21-scripts`.

In [ ]:
%%writefile /kaggle/working/scripts/run_all.py
#!/usr/bin/env python
"""Chạy tracker cho cả năm video (bản nộp) hoặc quét conf/iou (bản thử) trong một lệnh.

Script chỉ gọi lại ``run_tracking.py``; detector, kích thước ảnh và Re-ID vẫn cố định.
Cấu hình nộp nằm trong ``SUBMISSION_CONFIG`` — sửa sau khi đã xem video thử.

Ví dụ:
    # Bản nộp: đủ frame, đủ 5 video -> runs/nop_bai/video_N.txt
    python scripts/run_all.py --lab-data-root "$LAB_DATA" --device cuda:0

    # Bản thử: quét tracker x conf x iou, giới hạn frame, một video
    python scripts/run_all.py --lab-data-root "$LAB_DATA" --sweep \\
        --videos video_1 --max-frames 150
"""

from __future__ import annotations

import argparse
import itertools
import json
import subprocess
import sys
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

SCRIPT_DIR = Path(__file__).resolve().parent
VIDEOS = ["video_1", "video_2", "video_3", "video_4", "video_5"]

# video -> (tracker, conf, iou): cấu hình dự phòng khi chưa có best_config.json từ sweep.py.
#   video_1: HOTA cao nhất trong bảng quét (có nhãn).
#   video_2..5: không nhãn; botsort conf 0.15 có điểm thay thế (proxy) cao nhất ở cả bốn video
#   trong lần quét 150 frame đầu. Bản chạy đầy đủ sweep.py sẽ ghi đè bằng kết quả mịn hơn.
SUBMISSION_CONFIG: Dict[str, Tuple[str, float, float]] = {
    "video_1": ("botsort", 0.10, 0.5),
    "video_2": ("botsort", 0.15, 0.5),
    "video_3": ("botsort", 0.15, 0.5),
    "video_4": ("botsort", 0.15, 0.5),
    "video_5": ("botsort", 0.15, 0.5),
}

SWEEP_TRACKERS = ["bytetrack", "botsort"]
SWEEP_CONFS = [0.15, 0.3, 0.5]
SWEEP_IOUS = [0.4, 0.5, 0.7]


def build_command(
    lab_data_root: Path,
    video: str,
    tracker: str,
    conf: float,
    iou: float,
    out: Path,
    device: str,
    max_frames: int = 0,
    save_video: bool = True,
) -> List[str]:
    """Dựng lệnh gọi ``run_tracking.py`` cho một video.

    Args:
        lab_data_root: Thư mục lab_data giảng viên phát.
        video: Tên video, ví dụ ``video_1``.
        tracker: Tên tracker.
        conf: Ngưỡng confidence của detector.
        iou: Ngưỡng IoU NMS của detector.
        out: Thư mục xuất kết quả.
        device: ``cpu``, ``cuda:0``, ...
        max_frames: Giới hạn số frame; 0 nghĩa là toàn bộ.
        save_video: Có xuất video xem thử không.

    Returns:
        Danh sách tham số dòng lệnh, dùng được với ``subprocess.run``.
    """
    cmd = [
        sys.executable, str(SCRIPT_DIR / "run_tracking.py"),
        "--source", str(lab_data_root / video / "img1"),
        "--seq-name", video,
        "--tracker", tracker,
        "--conf", str(conf),
        "--iou", str(iou),
        "--out", str(out),
        "--device", device,
    ]
    if save_video:
        cmd.append("--save-video")
    if max_frames:
        cmd += ["--max-frames", str(max_frames)]
    return cmd


def merge_config(
    base: Dict[str, Tuple[str, float, float]], override: Dict[str, Sequence]
) -> Dict[str, Tuple[str, float, float]]:
    """Ghi đè cấu hình nộp bằng dict ``video -> [tracker, conf, iou]``.

    Args:
        base: Cấu hình gốc.
        override: Cấu hình mới, thường đọc từ file JSON (giá trị là list).

    Returns:
        Dict mới, không sửa ``base``.

    Raises:
        ValueError: Khi tên video không hợp lệ hoặc giá trị không đủ ba phần tử.
    """
    merged = dict(base)
    for video, value in override.items():
        if video not in base:
            raise ValueError(f"Video không hợp lệ: {video}")
        if len(value) != 3:
            raise ValueError(f"{video}: cần [tracker, conf, iou], nhận {value}")
        merged[video] = (str(value[0]), float(value[1]), float(value[2]))
    return merged


def sweep_out_dir(base: Path, tracker: str, conf: float, iou: float) -> Path:
    """Đặt tên thư mục kết quả cho một cấu hình quét.

    Args:
        base: Thư mục gốc chứa các lần thử.
        tracker: Tên tracker.
        conf: Ngưỡng confidence.
        iou: Ngưỡng IoU.

    Returns:
        Đường dẫn dạng ``base/<tracker>_c0.30_i0.5``.
    """
    return base / f"{tracker}_c{conf:.2f}_i{iou:g}"


def sweep_grid(
    trackers: Sequence[str], confs: Sequence[float], ious: Sequence[float]
) -> List[Tuple[str, float, float]]:
    """Liệt kê các cấu hình quét, mỗi lần chỉ đổi MỘT tham số quanh mốc (0.3, 0.5).

    Args:
        trackers: Các tracker cần thử.
        confs: Các giá trị ``conf``.
        ious: Các giá trị ``iou``.

    Returns:
        Danh sách ``(tracker, conf, iou)`` không trùng, giữ thứ tự.
    """
    base_conf, base_iou = 0.3, 0.5
    seen = []
    for tracker in trackers:
        for conf, iou in itertools.chain(
            ((c, base_iou) for c in confs), ((base_conf, i) for i in ious)
        ):
            item = (tracker, conf, iou)
            if item not in seen:
                seen.append(item)
    return seen


def main() -> None:
    """Chạy bản nộp cho các video, hoặc quét cấu hình khi có ``--sweep``."""
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--lab-data-root", required=True, type=Path)
    parser.add_argument("--device", default="cpu")
    parser.add_argument("--videos", nargs="+", default=VIDEOS, choices=VIDEOS)
    parser.add_argument("--out", type=Path, default=None, help="Mặc định runs/nop_bai (nộp) hoặc runs/thu_nghiem (quét)")
    parser.add_argument("--config-json", type=Path, default=None, help="File JSON ghi đè SUBMISSION_CONFIG")
    parser.add_argument("--sweep", action="store_true", help="Quét tracker/conf/iou thay vì chạy bản nộp")
    parser.add_argument("--max-frames", type=int, default=0, help="Chỉ dùng khi quét; bản nộp luôn đủ frame")
    args = parser.parse_args()

    jobs: List[List[str]] = []
    if args.sweep:
        base = args.out or Path("runs/thu_nghiem")
        for video in args.videos:
            for tracker, conf, iou in sweep_grid(SWEEP_TRACKERS, SWEEP_CONFS, SWEEP_IOUS):
                out = sweep_out_dir(base, tracker, conf, iou)
                jobs.append(build_command(
                    args.lab_data_root, video, tracker, conf, iou, out,
                    args.device, args.max_frames or 150,
                ))
    else:
        out = args.out or Path("runs/nop_bai")
        config = SUBMISSION_CONFIG
        if args.config_json:
            config = merge_config(config, json.loads(args.config_json.read_text(encoding="utf-8")))
        for video in args.videos:
            tracker, conf, iou = config[video]
            print(f"[cấu hình] {video}: {tracker} conf={conf} iou={iou}")
            jobs.append(build_command(args.lab_data_root, video, tracker, conf, iou, out, args.device))

    for i, cmd in enumerate(jobs, 1):
        print(f"\n=== [{i}/{len(jobs)}] {' '.join(cmd)}")
        subprocess.run(cmd, check=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/scripts/sweep.py
#!/usr/bin/env python
"""Quét tracker / conf / iou để chọn cấu hình nộp.

Hai chế độ:
    video1: chạy đủ frame, chấm HOTA / MOTA / IDF1 bằng TrackEval (video_1 có nhãn).
            Ba giai đoạn, mỗi lần chỉ đổi một tham số: tracker x conf -> conf mịn hơn
            (0.15, 0.5) cho tracker tốt nhất -> iou (0.4, 0.7) cho cấu hình tốt nhất.
    others: video_2..video_5 không có nhãn nên không chấm HOTA được. Dùng điểm thay thế
            ``proxy`` tính từ file kết quả (độ phủ hộp/frame và số chỗ track bị đứt quãng),
            với hệ số đã khớp trên 17 cấu hình của video_1 (xem ``PROXY_*``). Cũng ba giai đoạn
            như trên, rồi ghi cấu hình có ``proxy`` cao nhất.

``best_config.json`` được ghi/gộp theo từng video để ``run_all.py --config-json`` đọc.
Detector, kích thước ảnh và Re-ID vẫn cố định; chỉ đổi ``--tracker``, ``--conf``, ``--iou``.

Ví dụ:
    python scripts/sweep.py --mode video1 --lab-data-root "$LAB_DATA" \\
        --trackeval-root TrackEval --lab-eval-root lab_eval --device cuda:0
    python scripts/sweep.py --mode others --lab-data-root "$LAB_DATA" --device cuda:0
"""

from __future__ import annotations

import argparse
import csv
import json
import math
import os
import subprocess
import sys
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

SCRIPT_DIR = Path(__file__).resolve().parent
TRACKERS = ["bytetrack", "ocsort", "botsort", "strongsort", "deepocsort"]
CONFS_VIDEO1 = [0.1, 0.2, 0.3]
CONFS_OTHERS = [0.15, 0.3]
CONFS_REFINE_VIDEO1 = [0.15, 0.5]
CONFS_REFINE_OTHERS = [0.1, 0.2, 0.5]
IOUS_REFINE = [0.4, 0.7]
OTHER_VIDEOS = ["video_2", "video_3", "video_4", "video_5"]
SCORE_KEYS = ["HOTA", "DetA", "AssA", "MOTA", "IDF1", "IDSW"]
PROXY_FRAMES = 150

# Điểm thay thế cho video không nhãn: HOTA ước lượng = a + b * phủ_chuẩn_hoá + c * đứt_chuẩn_hoá.
# Hệ số khớp bằng bình phương tối thiểu trên 17 cấu hình của video_1 (150 frame đầu), trong đó
# phủ_chuẩn_hoá = hộp/frame chia cho giá trị lớn nhất cùng video, đứt_chuẩn_hoá = số chỗ track
# đứt quãng chia cho giá trị lớn nhất cùng video. Kiểm tra leave-one-out: tương quan 0,82 với HOTA
# thật, cấu hình được chọn đứng hạng 5/17 (HOTA kém cấu hình tốt nhất 0,8). Chỉ kiểm chứng trên một
# video có nhãn, nên chỉ dùng để xếp hạng, không xem là số HOTA.
PROXY_INTERCEPT = 24.53
PROXY_W_COVER = 13.35
PROXY_W_GAPS = -14.39


def parse_summary(text: str) -> Dict[str, float]:
    """Đọc file ``pedestrian_summary.txt`` của TrackEval (dòng tên cột, dòng giá trị).

    Args:
        text: Nội dung file.

    Returns:
        Dict tên cột -> giá trị số.

    Raises:
        ValueError: Khi file có ít hơn hai dòng hoặc số cột không khớp số giá trị.
    """
    lines = [line for line in text.strip().splitlines() if line.strip()]
    if len(lines) < 2:
        raise ValueError("Summary cần ít nhất hai dòng (tên cột, giá trị).")
    keys, values = lines[0].split(), lines[1].split()
    if len(keys) != len(values):
        raise ValueError(f"Số cột {len(keys)} khác số giá trị {len(values)}.")
    return {k: float(v) for k, v in zip(keys, values)}


def track_stats(lines: Sequence[str], n_frames: int) -> Dict[str, float]:
    """Thống kê nhanh một file kết quả MOT, không cần nhãn.

    Chỉ tính các dòng có ``frame <= n_frames``.

    Args:
        lines: Các dòng ``frame,id,x,y,w,h,conf,...``.
        n_frames: Số frame cần tính (và dùng làm mẫu số của hộp/frame).

    Returns:
        Dict gồm ``rows``, ``ids``, ``boxes_per_frame``, ``mean_track_len`` (số hộp trung bình
        trên mỗi ID) và ``gaps`` (số lần một ID bị đứt quãng: có frame trống ở giữa hai lần
        xuất hiện; đây là dấu hiệu mất rồi bắt lại hoặc đổi danh tính).
    """
    frames_by_id: Dict[int, List[int]] = {}
    rows = 0
    for line in lines:
        if not line.strip():
            continue
        parts = line.split(",")
        frame = int(float(parts[0]))
        if n_frames and frame > n_frames:
            continue
        frames_by_id.setdefault(int(float(parts[1])), []).append(frame)
        rows += 1
    gaps = 0
    for frames in frames_by_id.values():
        frames.sort()
        gaps += sum(1 for a, b in zip(frames, frames[1:]) if b - a > 1)
    ids = len(frames_by_id)
    return {
        "rows": rows,
        "ids": ids,
        "boxes_per_frame": rows / n_frames if n_frames else 0.0,
        "mean_track_len": rows / ids if ids else 0.0,
        "gaps": gaps,
    }


def proxy_scores(rows: Sequence[Dict[str, object]]) -> List[float]:
    """Tính điểm thay thế cho từng cấu hình của MỘT video (chuẩn hoá theo cấu hình tối đa).

    Args:
        rows: Mỗi hàng có ``boxes_per_frame`` và ``gaps``.

    Returns:
        Điểm HOTA ước lượng, cùng thứ tự với ``rows``. Giá trị chỉ dùng để xếp hạng.
    """
    if not rows:
        return []
    max_cover = max(float(r["boxes_per_frame"]) for r in rows) or 1.0
    max_gaps = max(float(r["gaps"]) for r in rows) or 1.0
    return [
        PROXY_INTERCEPT
        + PROXY_W_COVER * float(r["boxes_per_frame"]) / max_cover
        + PROXY_W_GAPS * float(r["gaps"]) / max_gaps
        for r in rows
    ]


def pearson(xs: Sequence[float], ys: Sequence[float]) -> float:
    """Hệ số tương quan Pearson.

    Args:
        xs: Dãy thứ nhất.
        ys: Dãy thứ hai, cùng độ dài.

    Returns:
        Hệ số trong [-1, 1]; ``0.0`` khi một dãy không đổi hoặc ít hơn hai phần tử.

    Raises:
        ValueError: Khi hai dãy khác độ dài.
    """
    if len(xs) != len(ys):
        raise ValueError("Hai dãy phải cùng độ dài.")
    n = len(xs)
    if n < 2:
        return 0.0
    mx, my = sum(xs) / n, sum(ys) / n
    sxx = sum((x - mx) ** 2 for x in xs)
    syy = sum((y - my) ** 2 for y in ys)
    if sxx == 0 or syy == 0:
        return 0.0
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / math.sqrt(sxx * syy)


def grid(trackers: Sequence[str], confs: Sequence[float], iou: float) -> List[Tuple[str, float, float]]:
    """Liệt kê cấu hình quét: mỗi tracker thử từng ``conf`` với một ``iou`` cố định.

    Args:
        trackers: Danh sách tracker.
        confs: Danh sách ``conf``.
        iou: Giá trị ``iou`` dùng chung.

    Returns:
        Danh sách ``(tracker, conf, iou)``.
    """
    return [(t, c, iou) for t in trackers for c in confs]


def format_table(rows: Sequence[Dict[str, object]], columns: Sequence[str]) -> str:
    """Dựng bảng Markdown từ danh sách dict.

    Args:
        rows: Mỗi phần tử là một hàng.
        columns: Thứ tự cột.

    Returns:
        Chuỗi Markdown có dòng tiêu đề và dòng phân cách.
    """
    def cell(value: object) -> str:
        return f"{value:.2f}" if isinstance(value, float) else str(value)

    out = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
    for row in rows:
        out.append("| " + " | ".join(cell(row.get(c, "")) for c in columns) + " |")
    return "\n".join(out)


def pick_best(rows: Sequence[Dict[str, object]], key: str = "HOTA") -> Dict[str, object]:
    """Chọn hàng có ``key`` lớn nhất.

    Args:
        rows: Các hàng kết quả đã chấm.
        key: Tên cột dùng để xếp hạng.

    Returns:
        Hàng tốt nhất.

    Raises:
        ValueError: Khi ``rows`` rỗng.
    """
    if not rows:
        raise ValueError("Không có kết quả nào để chọn.")
    return max(rows, key=lambda r: float(r[key]))


def merge_best_config(existing: Dict[str, list], video: str, row: Dict[str, object]) -> Dict[str, list]:
    """Gộp cấu hình tốt nhất của một video vào dict ``best_config``.

    Args:
        existing: Nội dung ``best_config.json`` hiện có (có thể rỗng).
        video: Tên video.
        row: Hàng kết quả có ``tracker``, ``conf``, ``iou``.

    Returns:
        Dict mới; các video khác giữ nguyên.
    """
    merged = dict(existing)
    merged[video] = [row["tracker"], row["conf"], row["iou"]]
    return merged


def run_tracker(lab_data: Path, video: str, tracker: str, conf: float, iou: float,
                out_dir: Path, device: str, max_frames: int) -> Path:
    """Gọi ``run_tracking.py`` cho một cấu hình và trả đường dẫn file kết quả.

    Args:
        lab_data: Thư mục lab_data.
        video: Tên video.
        tracker: Tên tracker.
        conf: Ngưỡng confidence của detector.
        iou: Ngưỡng IoU NMS của detector.
        out_dir: Thư mục xuất.
        device: ``cpu`` hoặc ``cuda:0``.
        max_frames: Giới hạn frame, 0 là đủ.

    Returns:
        Đường dẫn ``out_dir/<video>.txt``.

    Raises:
        RuntimeError: Khi ``run_tracking.py`` thoát với mã khác 0.
    """
    cmd = [
        sys.executable, str(SCRIPT_DIR / "run_tracking.py"),
        "--source", str(lab_data / video / "img1"), "--seq-name", video,
        "--tracker", tracker, "--conf", str(conf), "--iou", str(iou),
        "--out", str(out_dir), "--device", device,
    ]
    if max_frames:
        cmd += ["--max-frames", str(max_frames)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"{tracker} conf={conf} iou={iou} lỗi:\n{proc.stderr[-1500:]}")
    return out_dir / f"{video}.txt"


def score_video1(txt: Path, run_name: str, trackeval_root: Path, lab_eval_root: Path) -> Dict[str, float]:
    """Chấm video_1 bằng ``evaluate_practice.py`` rồi đọc bảng điểm.

    Args:
        txt: File ``video_1.txt`` cần chấm.
        run_name: Tên lần chấm (duy nhất cho mỗi cấu hình).
        trackeval_root: Thư mục TrackEval.
        lab_eval_root: Thư mục chứa ``video_1/gt``, ``seqinfo.ini``, ``eval_config.json``.

    Returns:
        Dict điểm TrackEval (HOTA, MOTA, IDF1, ...).

    Raises:
        RuntimeError: Khi chấm lỗi hoặc không tìm thấy file summary.
    """
    cmd = [
        sys.executable, str(SCRIPT_DIR / "evaluate_practice.py"),
        "--trackeval-root", str(trackeval_root), "--lab-data-root", str(lab_eval_root),
        "--submission", str(txt), "--run-name", run_name,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "MPLBACKEND": "Agg"})
    found = list((trackeval_root / "data" / "trackers").rglob(f"{run_name}/pedestrian_summary.txt"))
    if proc.returncode != 0 or not found:
        raise RuntimeError(f"Chấm {run_name} lỗi:\n{(proc.stderr or proc.stdout)[-1500:]}")
    return parse_summary(found[0].read_text())


def write_outputs(rows: Sequence[Dict[str, object]], columns: Sequence[str], out_dir: Path, stem: str) -> None:
    """Ghi bảng kết quả ra CSV và Markdown.

    Args:
        rows: Các hàng kết quả.
        columns: Thứ tự cột.
        out_dir: Thư mục xuất.
        stem: Tên file không có đuôi.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / f"{stem}.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(columns), extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    (out_dir / f"{stem}.md").write_text(format_table(rows, columns) + "\n", encoding="utf-8")


def update_best_config(ketqua: Path, video: str, row: Dict[str, object]) -> None:
    """Ghi cấu hình tốt nhất của ``video`` vào ``best_config.json`` (gộp với nội dung cũ).

    Args:
        ketqua: Thư mục kết quả.
        video: Tên video.
        row: Hàng kết quả tốt nhất.
    """
    path = ketqua / "best_config.json"
    existing = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    path.write_text(json.dumps(merge_best_config(existing, video, row)), encoding="utf-8")


def run_stage(configs: Sequence[Tuple[str, float, float]], seen: set, rows: List[Dict[str, object]],
              evaluate) -> None:
    """Chạy các cấu hình chưa thử trong ``configs`` bằng hàm ``evaluate``.

    Args:
        configs: Danh sách ``(tracker, conf, iou)``.
        seen: Tập cấu hình đã chạy; được cập nhật tại chỗ.
        rows: Danh sách kết quả; ``evaluate`` thêm hàng mới vào đây.
        evaluate: Hàm nhận ``(tracker, conf, iou)`` và thêm một hàng vào ``rows`` khi thành công.
    """
    for config in configs:
        key = (config[0], round(config[1], 3), round(config[2], 3))
        if key in seen:
            continue
        seen.add(key)
        evaluate(*config)


def sweep_video1(args: argparse.Namespace) -> None:
    """Quét ba giai đoạn cho video_1 và chấm HOTA bằng TrackEval.

    Args:
        args: Tham số dòng lệnh đã parse.
    """
    rows: List[Dict[str, object]] = []
    seen: set = set()

    def evaluate(tracker: str, conf: float, iou: float) -> None:
        name = f"{tracker}_c{conf:.2f}_i{iou:g}"
        out_dir = args.work / "thu_nghiem" / name
        print(f"[video_1] {name} ...", flush=True)
        try:
            txt = run_tracker(args.lab_data_root, "video_1", tracker, conf, iou, out_dir, args.device, 0)
            score = score_video1(txt, name, args.trackeval_root, args.lab_eval_root)
        except RuntimeError as err:
            print(f"  BỎ QUA: {err}", flush=True)
            return
        stats = track_stats(txt.read_text().splitlines(), PROXY_FRAMES)
        row = {"tracker": tracker, "conf": conf, "iou": iou, **{k: score.get(k, 0.0) for k in SCORE_KEYS},
               "bpf": stats["boxes_per_frame"], "gaps": stats["gaps"]}
        rows.append(row)
        print("  " + ", ".join(f"{k}={row[k]:.2f}" for k in SCORE_KEYS), flush=True)

    run_stage(grid(args.trackers, CONFS_VIDEO1, 0.5), seen, rows, evaluate)
    if not rows:
        raise SystemExit("Không cấu hình nào chấm được.")
    best = pick_best(rows)
    run_stage(grid([str(best["tracker"])], CONFS_REFINE_VIDEO1, 0.5), seen, rows, evaluate)
    best = pick_best(rows)
    run_stage([(str(best["tracker"]), float(best["conf"]), iou) for iou in IOUS_REFINE], seen, rows, evaluate)

    for row, score in zip(rows, proxy_scores([{"boxes_per_frame": r["bpf"], "gaps": r["gaps"]} for r in rows])):
        row["proxy"] = score
    corr = pearson([float(r["proxy"]) for r in rows], [float(r["HOTA"]) for r in rows])
    rows.sort(key=lambda r: -float(r["HOTA"]))
    columns = ["tracker", "conf", "iou"] + SCORE_KEYS + ["bpf", "gaps", "proxy"]
    write_outputs(rows, columns, args.ketqua, "sweep_video_1")
    best = pick_best(rows)
    update_best_config(args.ketqua, "video_1", best)
    print("\n" + format_table(rows, columns))
    print(f"\nTương quan điểm thay thế (proxy) với HOTA thật trên {len(rows)} cấu hình: {corr:+.2f}")
    print(f"Tốt nhất theo HOTA: {best['tracker']} conf={best['conf']} iou={best['iou']}")


def sweep_others(args: argparse.Namespace) -> None:
    """Quét ba giai đoạn cho video_2..5 và xếp hạng bằng điểm thay thế ``proxy``.

    Args:
        args: Tham số dòng lệnh đã parse.
    """
    for video in args.videos:
        rows: List[Dict[str, object]] = []
        seen: set = set()

        def evaluate(tracker: str, conf: float, iou: float, video: str = video) -> None:
            name = f"{tracker}_c{conf:.2f}_i{iou:g}"
            print(f"[{video}] {name} ...", flush=True)
            try:
                txt = run_tracker(args.lab_data_root, video, tracker, conf, iou,
                                  args.work / "thu_nghiem" / video / name, args.device, args.max_frames)
            except RuntimeError as err:
                print(f"  BỎ QUA: {err}", flush=True)
                return
            stats = track_stats(txt.read_text().splitlines(), args.max_frames)
            rows.append({"tracker": tracker, "conf": conf, "iou": iou, **stats})

        def rank() -> Dict[str, object]:
            for row, score in zip(rows, proxy_scores(rows)):
                row["proxy"] = score
            return pick_best(rows, "proxy")

        run_stage(grid(args.trackers, CONFS_OTHERS, 0.5), seen, rows, evaluate)
        if not rows:
            print(f"[{video}] không cấu hình nào chạy được, bỏ qua.")
            continue
        best = rank()
        run_stage(grid([str(best["tracker"])], CONFS_REFINE_OTHERS, 0.5), seen, rows, evaluate)
        best = rank()
        run_stage([(str(best["tracker"]), float(best["conf"]), iou) for iou in IOUS_REFINE], seen, rows, evaluate)
        best = rank()

        rows.sort(key=lambda r: -float(r["proxy"]))
        columns = ["tracker", "conf", "iou", "rows", "ids", "boxes_per_frame", "mean_track_len", "gaps", "proxy"]
        write_outputs(rows, columns, args.ketqua, f"sweep_{video}")
        update_best_config(args.ketqua, video, best)
        print(f"\n### {video} ({args.max_frames} frame)\n" + format_table(rows, columns) + "\n")
        print(f"Tốt nhất theo proxy: {best['tracker']} conf={best['conf']} iou={best['iou']}\n")


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    """Khai báo tham số dòng lệnh.

    Args:
        argv: Danh sách tham số; ``None`` nghĩa là đọc từ ``sys.argv``.

    Returns:
        Namespace chứa các tham số đã parse.
    """
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--mode", required=True, choices=["video1", "others"])
    parser.add_argument("--lab-data-root", required=True, type=Path)
    parser.add_argument("--trackeval-root", type=Path, default=Path("TrackEval"))
    parser.add_argument("--lab-eval-root", type=Path, default=Path("lab_eval"))
    parser.add_argument("--device", default="cpu")
    parser.add_argument("--work", type=Path, default=Path("runs"), help="Thư mục chứa kết quả từng lần thử")
    parser.add_argument("--ketqua", type=Path, default=Path("ketqua"), help="Thư mục ghi bảng tổng hợp")
    parser.add_argument("--trackers", nargs="+", default=TRACKERS, choices=TRACKERS)
    parser.add_argument("--videos", nargs="+", default=OTHER_VIDEOS, choices=OTHER_VIDEOS)
    parser.add_argument("--max-frames", type=int, default=PROXY_FRAMES, help="Chỉ dùng cho chế độ others")
    return parser.parse_args(argv)


if __name__ == "__main__":
    arguments = parse_args()
    arguments.ketqua.mkdir(parents=True, exist_ok=True)
    if arguments.mode == "video1":
        sweep_video1(arguments)
    else:
        sweep_others(arguments)

In [ ]:
# 4. Dựng thư mục chấm video_1 (nhãn + cấu hình). Dữ liệu trong /kaggle/input chỉ đọc và có thể thiếu
#    eval_config.json / seqinfo.ini, nên dựng bản sao nhỏ trong /kaggle/working.
import json, cv2
os.environ["MPLBACKEND"] = "Agg"  # kernel đặt backend inline, tiến trình con (matplotlib trong venv) không hiểu
KETQUA = WORK / "ketqua"
KETQUA.mkdir(exist_ok=True)
src = Path(os.environ["LAB_DATA"]) / "video_1"
dst = WORK / "lab_eval" / "video_1"
(dst / "gt").mkdir(parents=True, exist_ok=True)
shutil.copy(src / "gt" / "gt.txt", dst / "gt" / "gt.txt")

if (src / "eval_config.json").exists():
    shutil.copy(src / "eval_config.json", dst / "eval_config.json")
else:
    print("Gói dữ liệu không có eval_config.json -> dùng benchmark MOT17 (định dạng MOTChallenge).")
    (dst / "eval_config.json").write_text(json.dumps({"benchmark": "MOT17", "split": "train"}))

if (src / "seqinfo.ini").exists():
    shutil.copy(src / "seqinfo.ini", dst / "seqinfo.ini")
else:
    imgs = sorted((src / "img1").glob("*.jpg"))
    h, w = cv2.imread(str(imgs[0])).shape[:2]
    NL = chr(10)  # tránh ký tự backslash trong chuỗi
    lines = ["[Sequence]", "name=video_1", "imDir=img1", "frameRate=30",
             f"seqLength={len(imgs)}", f"imWidth={w}", f"imHeight={h}", "imExt=.jpg"]
    (dst / "seqinfo.ini").write_text(NL.join(lines) + NL)
print("lab_eval sẵn sàng:", sorted(p.name for p in dst.iterdir()))

In [ ]:
# 5. QUÉT video_1 (có nhãn, chấm HOTA): 5 tracker x conf {0.1, 0.2, 0.3}; rồi conf {0.15, 0.5} cho tracker tốt nhất;
#    rồi iou {0.4, 0.7} cho cấu hình tốt nhất (mỗi lần chỉ đổi một tham số). Ghi ketqua/sweep_video_1.md|csv và best_config.json.
RUN_SWEEP_VIDEO1 = True  # ~30 phút. Đặt False để bỏ qua (khi đó video_1 dùng cấu hình dự phòng trong run_all.py)
if RUN_SWEEP_VIDEO1:
    !$PY scripts/sweep.py --mode video1 --lab-data-root "$LAB_DATA" --trackeval-root /kaggle/working/TrackEval --lab-eval-root /kaggle/working/lab_eval --device "$DEVICE" --work /kaggle/working/runs --ketqua /kaggle/working/ketqua

In [ ]:
# 6. QUÉT video_2..5 (không nhãn), 150 frame đầu, cùng ba giai đoạn như cell 5. Xếp hạng bằng điểm thay thế
#    (proxy) = độ phủ hộp/frame và số chỗ track bị đứt quãng, hệ số khớp trên video_1 (tương quan với HOTA thật in ở cell 5).
#    Ghi ketqua/sweep_video_N.md|csv và gộp vào best_config.json.
RUN_SWEEP_OTHERS = True  # ~35 phút. Đặt False để bỏ qua (khi đó video_2..5 dùng cấu hình dự phòng trong run_all.py)
if RUN_SWEEP_OTHERS:
    !$PY scripts/sweep.py --mode others --lab-data-root "$LAB_DATA" --device "$DEVICE" --work /kaggle/working/runs --ketqua /kaggle/working/ketqua --max-frames 150

In [ ]:
# 7. BẢN NỘP: đủ frame, đủ 5 video -> runs/nop_bai/video_N.txt
#    Cấu hình lấy từ ketqua/best_config.json (do cell 5 và 6 ghi); video nào chưa có thì dùng SUBMISSION_CONFIG
#    trong run_all.py. Muốn ép một video: sửa OVERRIDE, ví dụ {"video_2": ["ocsort", 0.2, 0.5]}.
OVERRIDE = {}
cfg = {}
best_file = KETQUA / "best_config.json"
if best_file.exists():
    cfg.update(json.loads(best_file.read_text()))
cfg.update(OVERRIDE)
(KETQUA / "final_config.json").write_text(json.dumps(cfg))
print("cấu hình ghi đè:", cfg)
!$PY scripts/run_all.py --lab-data-root "$LAB_DATA" --device "$DEVICE" --config-json /kaggle/working/ketqua/final_config.json
!ls -la runs/nop_bai

In [ ]:
# 8. Chấm video_1 bản nộp (HOTA / MOTA / IDF1) — dán bảng này vào BAO_CAO.md
!$PY scripts/evaluate_practice.py --trackeval-root /kaggle/working/TrackEval --lab-data-root /kaggle/working/lab_eval --submission runs/nop_bai/video_1.txt --run-name final_video1 2>&1 | tee /kaggle/working/ketqua/danh_gia_video_1.txt

In [ ]:
# 9. Gom kết quả vào /kaggle/working/ketqua và nén
import zipfile

OUT = KETQUA
RUNS = WORK / "runs" / "nop_bai"

# --- đối chiếu số frame với số ảnh từng video
for i in range(1, 6):
    p = RUNS / f"video_{i}.txt"
    n_imgs = len(list((Path(os.environ["LAB_DATA"]) / f"video_{i}" / "img1").glob("*.jpg")))
    frames = {int(l.split(",")[0]) for l in p.read_text().splitlines() if l.strip()} if p.exists() else set()
    print(f"video_{i}: {'OK ' if p.exists() else 'THIẾU'} | {n_imgs} ảnh | frame có track: {len(frames)} | frame cuối: {max(frames) if frames else 0}")

# --- bảng điểm bản nộp của video_1
summary = list((WORK / "TrackEval" / "data" / "trackers").rglob("final_video1/pedestrian_summary.txt"))
if summary:
    shutil.copy(summary[0], OUT / "pedestrian_summary_video_1.txt")
    print("Điểm video_1:")
    print((OUT / "pedestrian_summary_video_1.txt").read_text())

# --- các file tổng hợp nhỏ (bảng quét, cấu hình, điểm)
small = [p for p in sorted(OUT.iterdir()) if p.suffix in {".txt", ".md", ".csv", ".json"}]

# --- gói 1: BÀI NỘP nhỏ (5 file video_N.txt + bảng quét + điểm)
sub_zip = OUT / "nop_bai.zip"
with zipfile.ZipFile(sub_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUNS.glob("video_*.txt")):
        z.write(p, f"nop_bai/{p.name}")
    for p in small:
        z.write(p, f"nop_bai/{p.name}")

# --- gói 2: TẤT CẢ trong MỘT file zip (thêm 5 video xem thử để điền cột Quan sát)
all_zip = OUT / "ket_qua_lab21.zip"
with zipfile.ZipFile(all_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUNS.glob("video_*.txt")):
        z.write(p, f"ket_qua_lab21/nop_bai/{p.name}")
    for p in small:
        z.write(p, f"ket_qua_lab21/{p.name}")
    for p in sorted(RUNS.glob("*_preview.mp4")):
        z.write(p, f"ket_qua_lab21/video_xem_thu/{p.name}")
for f in (sub_zip, all_zip):
    print(f"{f.name}: {f.stat().st_size / 1e6:.1f} MB")

In [ ]:
# 10. (Tùy chọn) Dọn thư mục nặng trước khi bấm Save Version, để Output chỉ còn /kaggle/working/ketqua
#     Đặt True rồi chạy. Sau khi dọn phải chạy lại từ cell 1 nếu muốn chạy tiếp.
CLEANUP = False
if CLEANUP:
    for name in ("venv", "TrackEval", "lab_eval", "scripts", "runs", "lab_raw"):
        shutil.rmtree(WORK / name, ignore_errors=True)
    for f in WORK.glob("*.pt"):
        f.unlink()
    print("Đã dọn. Còn lại:", sorted(p.name for p in WORK.iterdir()))

In [ ]:
# 11. TẢI VỀ MÁY: bấm vào link bên dưới (hoặc khung Output bên phải -> ketqua -> dấu ba chấm -> Download)
from IPython.display import FileLink, display
for name in ("nop_bai.zip", "ket_qua_lab21.zip"):
    f = WORK / "ketqua" / name
    if f.exists():
        print(f"{name}  ({f.stat().st_size / 1e6:.1f} MB)")
        display(FileLink(f"ketqua/{name}"))
    else:
        print(f"THIẾU {name}: chạy lại cell 9")
print("nop_bai.zip = 5 file video_N.txt + bảng quét + điểm (nộp bài).")
print("ket_qua_lab21.zip = tất cả, kèm 5 video xem thử để điền cột Quan sát.")